In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import gurobipy as gp
from gurobipy import GRB

# Clean module imports using the src package
from src.data_utils import parse_file_data, get_old_voltages, get_edges
from src.formulation import build_mp_ac_uc_rectangular, build_convex_ac_uc, _precompute_bus_connectivity

# Formatting for Pandas
pd.set_option('display.float_format', lambda x: '%.4f' % x)

In [2]:
# 1. Load the IEEE test case
# Make sure to point to wherever your case14.m file is located! 
# If it is inside src, use 'src/case14.m'
file_path = 'data/case14.m' 
data = parse_file_data(file_path)

print(f"Loaded Grid: {len(data.buses)} Buses, {len(data.gens)} Generators")

# 2. Solve the True Non-Convex AC-OPF at 100% load to get the V_base expansion points
print("Solving base non-convex case to extract Taylor expansion points (V_base)...")
base_true_model = build_mp_ac_uc_rectangular(data, demand_curve=[1.0])
base_true_model.setParam('OutputFlag', 0)
base_true_model.optimize()

if base_true_model.status == GRB.OPTIMAL:
    # Extract the base voltages for the Can Li QCAC relaxation
    node_vr, node_vi = get_old_voltages(base_true_model, data)
    print("Successfully extracted V_base warm-start points.")
else:
    raise RuntimeError("Base non-convex model failed to solve. Check grid data.")

Loaded Grid: 14 Buses, 5 Generators
Solving base non-convex case to extract Taylor expansion points (V_base)...
Set parameter Username
Academic license - for non-commercial use only - expires 2027-01-17
Set parameter NonConvex to value 2
Successfully extracted V_base warm-start points.


In [3]:
# Define 5 scenarios: Scaling the total active and reactive demand
load_scales = [0.8, 0.9, 1.0, 1.1, 1.2]
results = []

# Precompute base loads for the convex model
conn = _precompute_bus_connectivity(data)
base_pd = {i: conn[i]['base_pd'] for i in data.buses}
base_qd = {i: conn[i]['base_qd'] for i in data.buses}
gen_ids_sorted = sorted(list(data.gens.keys()))

print(f"{'Scale':<10} | {'Model':<15} | {'Status':<10} | {'Obj Cost':<12} | {'Time (s)':<10} | {'Commitment (u)'}")
print("-" * 85)

for scale in load_scales:
    
    # -----------------------------------------------------------------
    # 1. Solve ORIGINAL MINLP (Non-Convex MIQCQP)
    # -----------------------------------------------------------------
    t0 = time.time()
    # We use mp_ac_uc with T=1 and our specific load multiplier
    model_true = build_mp_ac_uc_rectangular(data, demand_curve=[scale])
    model_true.setParam('OutputFlag', 0)
    model_true.setParam('TimeLimit', 60.0)
    model_true.optimize()
    t_true = time.time() - t0
    
    if model_true.status in [GRB.OPTIMAL, GRB.SUBOPTIMAL]:
        obj_true = model_true.ObjVal
        u_true = [int(round(model_true._u[g, 1].X)) for g in gen_ids_sorted]
        status_true = "Optimal"
    else:
        obj_true = np.nan
        u_true = ["-"] * len(gen_ids_sorted)
        status_true = "Failed"
        
    print(f"{scale:<10} | {'True MIQCQP':<15} | {status_true:<10} | {obj_true:<12.2f} | {t_true:<10.4f} | {u_true}")

    # -----------------------------------------------------------------
    # 2. Solve CAN LI RELAXATION (Convex QCAC MIQCQP)
    # -----------------------------------------------------------------
    t0 = time.time()
    # Scale the absolute loads for the convex builder
    current_pd = {i: base_pd[i] * scale for i in data.buses}
    current_qd = {i: base_qd[i] * scale for i in data.buses}
    
    model_relax = build_convex_ac_uc(
        data, node_vr, node_vi, 
        node_pd=current_pd, node_qd=current_qd, conn=conn
    )
    model_relax.setParam('OutputFlag', 0)
    model_relax.optimize()
    t_relax = time.time() - t0
    
    if model_relax.status in [GRB.OPTIMAL, GRB.SUBOPTIMAL]:
        obj_relax = model_relax.ObjVal
        u_relax = [int(round(model_relax._u[g, 1].X)) for g in gen_ids_sorted]
        status_relax = "Optimal"
        
        # Calculate maximum slack violation to see if the convexification holds
        max_slack = max([model_relax._xi_c[i,1].X for i in data.buses] + 
                        [model_relax._xij_c[e[0], e[1], 1].X for e in get_edges(data)] +
                        [model_relax._xij_s[e[0], e[1], 1].X for e in get_edges(data)])
    else:
        obj_relax = np.nan
        u_relax = ["-"] * len(gen_ids_sorted)
        status_relax = "Failed"
        max_slack = np.nan
        
    print(f"{scale:<10} | {'QCAC Convex':<15} | {status_relax:<10} | {obj_relax:<12.2f} | {t_relax:<10.4f} | {u_relax}")
    print("-" * 85)
    
    # Store results
    results.append({
        'Load_Scale': scale,
        'True_Obj': obj_true,
        'Relax_Obj': obj_relax,
        'Gap_%': abs(obj_true - obj_relax)/obj_true * 100 if status_true == "Optimal" else np.nan,
        'True_Time': t_true,
        'Relax_Time': t_relax,
        'Max_Slack': max_slack,
        'U_Match': u_true == u_relax
    })

df_results = pd.DataFrame(results)

Scale      | Model           | Status     | Obj Cost     | Time (s)   | Commitment (u)
-------------------------------------------------------------------------------------
Set parameter NonConvex to value 2
0.8        | True MIQCQP     | Optimal    | 1085.31      | 19.1827    | [1, 1, 1, 1, 1]
0.8        | QCAC Convex     | Optimal    | 1757.45      | 0.0763     | [1, 1, 1, 1, 1]
-------------------------------------------------------------------------------------
Set parameter NonConvex to value 2
0.9        | True MIQCQP     | Optimal    | 1462.62      | 7.5151     | [1, 1, 1, 1, 1]
0.9        | QCAC Convex     | Failed     | nan          | 0.8235     | ['-', '-', '-', '-', '-']
-------------------------------------------------------------------------------------
Set parameter NonConvex to value 2
1.0        | True MIQCQP     | Optimal    | 2190.26      | 8.9073     | [1, 1, 1, 1, 1]
1.0        | QCAC Convex     | Optimal    | 2190.99      | 0.0283     | [1, 1, 1, 1, 1]
------------